# 00 — Data Exploration
**MM-WHS Dataset · 2D slice format (256×256 NPZ)**

Covers:
- Split summary (patients / slices per modality)
- Label class distribution
- Image intensity statistics
- Sample slice visualisations
- DataLoader speed benchmark

In [ ]:
import sys, os

# Always run from project root regardless of how the notebook is invoked
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import time
from pathlib import Path

from src.data.mmwhs_dataset import (
    MMWHSSliceDataset, MMWHSPatientDataset,
    make_dataloaders, LABEL_NAMES, NUM_CLASSES
)

DATA_DIR = 'data/pack/processed_data'
COLORS = plt.get_cmap('tab10', NUM_CLASSES)
print(f'Working dir: {os.getcwd()}')
print('Ready.')

## 1 · Split Summary

In [ ]:
rows = []
for mod in ('ct', 'mr'):
    for split in ('train', 'val', 'test'):
        ds = MMWHSSliceDataset(DATA_DIR, mod, split, preload=False)
        patients = ds.get_patients()
        rows.append((mod.upper(), split, len(ds), len(patients), ', '.join(patients)))

print(f"{'Mod':<5} {'Split':<7} {'Slices':>7} {'Patients':>9}  Patient IDs")
print('-' * 75)
for mod, split, n_slices, n_pat, pids in rows:
    print(f'{mod:<5} {split:<7} {n_slices:>7,} {n_pat:>9}  {pids}')

## 2 · Class Label Distribution

In [ ]:
def count_labels(modality, split='train'):
    npz_dir = Path(DATA_DIR) / f'{modality}_256' / split / 'npz'
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for f in sorted(npz_dir.glob('*.npz')):
        lbl = np.load(f)['label']
        for c in range(NUM_CLASSES):
            counts[c] += (lbl == c).sum()
    return counts

print('Computing label distributions (train sets)...')
ct_counts  = count_labels('ct',  'train')
mr_counts  = count_labels('mr',  'train')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_labels = [LABEL_NAMES[c] for c in range(NUM_CLASSES)]
x = np.arange(NUM_CLASSES)

for ax, counts, title in zip(axes, [ct_counts, mr_counts], ['CT Train', 'MRI Train']):
    pcts = 100 * counts / counts.sum()
    bars = ax.bar(x, pcts, color=[COLORS(c) for c in range(NUM_CLASSES)], edgecolor='black', linewidth=0.5)
    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('% of total pixels')
    ax.set_title(f'{title} — Label Distribution')
    ax.set_ylim(0, max(pcts) * 1.15)

plt.tight_layout()
plt.savefig('results/00_label_distribution.png', dpi=120)
plt.show()
print('Saved: results/00_label_distribution.png')

In [ ]:
# Foreground-only view (exclude background)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, counts, title in zip(axes, [ct_counts, mr_counts], ['CT Train', 'MRI Train']):
    fg_counts = counts[1:]
    pcts = 100 * fg_counts / fg_counts.sum()
    fg_labels = [LABEL_NAMES[c] for c in range(1, NUM_CLASSES)]
    bars = ax.bar(np.arange(NUM_CLASSES - 1), pcts,
                  color=[COLORS(c) for c in range(1, NUM_CLASSES)],
                  edgecolor='black', linewidth=0.5)
    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(np.arange(NUM_CLASSES - 1))
    ax.set_xticklabels(fg_labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('% of foreground pixels')
    ax.set_title(f'{title} — Foreground Class Distribution')

plt.tight_layout()
plt.savefig('results/00_foreground_distribution.png', dpi=120)
plt.show()

## 3 · Image Intensity Statistics

In [ ]:
def sample_intensities(modality, split='train', n_slices=200):
    npz_dir = Path(DATA_DIR) / f'{modality}_256' / split / 'npz'
    files = sorted(npz_dir.glob('*.npz'))
    step = max(1, len(files) // n_slices)
    imgs = [np.load(f)['image'].astype(np.float32).ravel() for f in files[::step]]
    return np.concatenate(imgs)

print('Sampling intensities...')
ct_px  = sample_intensities('ct')
mr_px  = sample_intensities('mr')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, px, title in zip(axes, [ct_px, mr_px], ['CT', 'MRI']):
    ax.hist(px, bins=100, color='steelblue', edgecolor='none', alpha=0.85)
    ax.axvline(px.mean(), color='red', lw=1.5, label=f'mean={px.mean():.3f}')
    ax.axvline(np.percentile(px, 1),  color='orange', lw=1, ls='--', label='p1')
    ax.axvline(np.percentile(px, 99), color='orange', lw=1, ls='--', label='p99')
    ax.set_xlabel('Normalised intensity')
    ax.set_ylabel('Pixel count')
    ax.set_title(f'{title} Intensity Distribution (sample of 200 slices)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/00_intensity_distribution.png', dpi=120)
plt.show()

for px, title in [(ct_px, 'CT'), (mr_px, 'MRI')]:
    print(f'{title}: min={px.min():.3f}  p1={np.percentile(px,1):.3f}  '
          f'mean={px.mean():.3f}  p99={np.percentile(px,99):.3f}  max={px.max():.3f}')

## 4 · Sample Slice Visualisation

In [ ]:
def pick_rich_slices(modality, split='train', n=6, min_classes=4):
    """Return n slices that contain at least min_classes foreground classes."""
    ds = MMWHSSliceDataset(DATA_DIR, modality, split, preload=True)
    picked = []
    for i, sample in enumerate(ds):
        lbl = sample['label'].numpy()
        if len(np.unique(lbl)) >= min_classes + 1:  # +1 for BG
            picked.append(sample)
        if len(picked) == n:
            break
    return picked

legend_patches = [mpatches.Patch(color=COLORS(i), label=f'{i}: {LABEL_NAMES[i]}') for i in range(NUM_CLASSES)]

for modality in ('ct', 'mr'):
    samples = pick_rich_slices(modality, n=6)
    fig = plt.figure(figsize=(18, 7))
    fig.suptitle(f'{modality.upper()} — Representative Slices (image | label overlay | label only)',
                 fontsize=12, fontweight='bold')
    gs = gridspec.GridSpec(2, len(samples), figure=fig, hspace=0.05, wspace=0.03)

    for col, s in enumerate(samples):
        img = s['image'].squeeze().numpy()
        lbl = s['label'].numpy()

        ax_img = fig.add_subplot(gs[0, col])
        ax_img.imshow(img, cmap='gray')
        ax_img.set_title(s['patient'], fontsize=7)
        ax_img.axis('off')

        ax_ov = fig.add_subplot(gs[1, col])
        ax_ov.imshow(img, cmap='gray')
        rgba = COLORS(lbl / NUM_CLASSES)
        rgba[..., 3] = (lbl > 0).astype(float) * 0.65
        ax_ov.imshow(rgba)
        ax_ov.axis('off')

    fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=8,
               bbox_to_anchor=(0.5, -0.04))
    out = f'results/00_samples_{modality}.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## 5 · Per-Patient Slice Count & Label Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, modality in zip(axes, ('ct', 'mr')):
    patient_ds = MMWHSPatientDataset(DATA_DIR, modality, 'train')
    data = []
    for i in range(len(patient_ds)):
        s = patient_ds[i]
        lbl_vol = s['label'].numpy()
        classes_present = [c for c in range(1, NUM_CLASSES) if (lbl_vol == c).any()]
        data.append((s['patient'], s['n_slices'], classes_present))

    patients = [d[0] for d in data]
    n_slices = [d[1] for d in data]
    n_classes = [len(d[2]) for d in data]

    x = np.arange(len(patients))
    bars = ax.bar(x, n_slices, color='steelblue', alpha=0.8, label='# slices')
    ax2 = ax.twinx()
    ax2.plot(x, n_classes, 'ro-', ms=5, label='# fg classes')
    ax2.set_ylabel('Foreground classes present', color='red')
    ax2.tick_params(axis='y', colors='red')
    ax2.set_ylim(0, 8)

    ax.set_xticks(x)
    ax.set_xticklabels([p.split('_')[1] for p in patients], rotation=45, ha='right')
    ax.set_xlabel('Patient ID')
    ax.set_ylabel('Number of slices', color='steelblue')
    ax.set_title(f'{modality.upper()} Train — Slices & Class Coverage per Patient')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('results/00_patient_stats.png', dpi=120)
plt.show()

## 6 · DataLoader Speed Benchmark

In [ ]:
print('Benchmarking DataLoader (preload=True vs preload=False)...')
results = {}
for preload in (True, False):
    for modality in ('ct', 'mr'):
        ds = MMWHSSliceDataset(DATA_DIR, modality, 'train', preload=preload)
        from torch.utils.data import DataLoader
        loader = DataLoader(ds, batch_size=16, shuffle=True)

        t0 = time.time()
        for i, batch in enumerate(loader):
            _ = batch['image'], batch['label']
            if i == 9:
                break
        elapsed = time.time() - t0
        key = f'{modality.upper()} preload={preload}'
        results[key] = elapsed / 10 * 1000
        print(f'  {key}: {elapsed/10*1000:.1f} ms/batch')

print('\nSummary:')
for k, v in results.items():
    print(f'  {k:<30} {v:>8.1f} ms/batch')

## 7 · Class Weights (computed for training)

In [ ]:
import torch

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, modality in zip(axes, ('ct', 'mr')):
    w = torch.load(f'data/class_weights_{modality}.pt', weights_only=True).numpy()
    x = np.arange(NUM_CLASSES)
    bars = ax.bar(x, w, color=[COLORS(c) for c in range(NUM_CLASSES)], edgecolor='black', lw=0.5)
    for bar, wi in zip(bars, w):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{wi:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([LABEL_NAMES[i] for i in range(NUM_CLASSES)], rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Weight')
    ax.set_title(f'{modality.upper()} — Inverse-Frequency Class Weights')
    ax.axhline(1.0, color='grey', ls='--', lw=0.8)

plt.tight_layout()
plt.savefig('results/00_class_weights.png', dpi=120)
plt.show()

## Summary

| Observation | Detail |
|---|---|
| Data format | 2D NPZ slices (256×256), already normalised |
| CT splits | 3389 train / 382 val / 484 test slices · 16/2/2 patients |
| MRI splits | 1738 train / 254 val / 236 test slices · 16/2/2 patients |
| Class imbalance | Background 88–94%; each structure 0.4–2.3% |
| Preload speedup | ~300× faster batches (4ms vs 1280ms) |
| Hardest structure | PA has smallest foreground fraction → most likely to under-segment |